# Same-dataset benchmark — YOLO26s baseline

Benchmark tái lập giao thức công khai của paper 2025 trên `No Helmet No Ride`. Notebook này chỉ đánh giá `valid` (355 ảnh); folder `test` 101 ảnh được khóa và không được sử dụng.

**An toàn khi mở:** `RUN_TRAINING`, `RUN_VALIDATION` và `RUN_COMPLEXITY` đều mặc định `False`. Không có training nào được chạy khi tạo hoặc mở notebook.

In [ ]:
%pip install -q ultralytics==8.4.155 pyyaml pandas matplotlib seaborn


In [ ]:
from pathlib import Path
from collections import Counter
import inspect
import math
import random
import time

import numpy as np
import pandas as pd
import torch
import yaml
import ultralytics

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    print('Không chạy trong Google Colab; dùng các đường dẫn local nếu có.')

ULTRALYTICS_VERSION = '8.4.155'
MODEL_NAME = 'yolo26s.pt'
EXPECTED_DATASET_NAME = 'No Helmet No Ride'
EXPECTED_CLASS_NAMES = ['helmet', 'nohelmet', 'riders']
EXPECTED_TRAIN_IMAGES = 3912
EXPECTED_VALID_IMAGES = 355
EXPECTED_TEST_IMAGES = 101
IMG_SIZE = 640
RANDOM_SEED = 0

# Chỉ đổi các cờ này khi chủ động chạy một giai đoạn sau audit.
RUN_TRAINING = False
RUN_VALIDATION = False
RUN_COMPLEXITY = False
DATASET_ROOT_OVERRIDE = None  # Ví dụ: '/content/drive/MyDrive/NO Helmet NO Ride.v2i.yolov8'

assert ultralytics.__version__ == ULTRALYTICS_VERSION, ultralytics.__version__
SEARCH_ROOTS = [Path('/content/drive/MyDrive'), Path('/content'), Path.cwd()]
DATASET_CANDIDATE_NAMES = ['NO Helmet NO Ride.v2i.yolov8', 'NO Helmet NO Ride']

def locate_dataset():
    if DATASET_ROOT_OVERRIDE:
        root = Path(DATASET_ROOT_OVERRIDE).expanduser().resolve()
        if not (root / 'data.yaml').exists():
            raise FileNotFoundError(f'Không thấy data.yaml tại {root}')
        return root
    for search_root in SEARCH_ROOTS:
        for name in DATASET_CANDIDATE_NAMES:
            candidate = search_root / name
            if (candidate / 'data.yaml').exists():
                return candidate.resolve()
            nested = search_root / 'DetectedHelmets' / name
            if (nested / 'data.yaml').exists():
                return nested.resolve()
    raise FileNotFoundError('Không tìm thấy dataset No Helmet No Ride; đặt DATASET_ROOT_OVERRIDE.')

DATASET_ROOT = locate_dataset()
DATA_YAML_PATH = DATASET_ROOT / 'data.yaml'
with DATA_YAML_PATH.open('r', encoding='utf-8') as handle:
    DATA_CONFIG = yaml.safe_load(handle)

def normalize_names(raw_names):
    if isinstance(raw_names, dict):
        return [raw_names[key] for key in sorted(raw_names, key=lambda item: int(item))]
    return list(raw_names or [])

CLASS_NAMES = normalize_names(DATA_CONFIG.get('names'))
assert DATA_CONFIG.get('nc') == 3, DATA_CONFIG.get('nc')
assert CLASS_NAMES == EXPECTED_CLASS_NAMES, CLASS_NAMES

def resolve_split_dir(split_name):
    raw_value = DATA_CONFIG.get(split_name)
    if not raw_value:
        raise KeyError(f'data.yaml không có split {split_name!r}')
    raw_path = Path(str(raw_value))
    candidates = [
        raw_path if raw_path.is_absolute() else DATA_YAML_PATH.parent / raw_path,
        DATASET_ROOT / split_name / 'images',
        DATA_YAML_PATH.parent.parent / split_name / 'images',
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(f'Không resolve được {split_name}: {candidates}')

SPLIT_DIRS = {split_name: resolve_split_dir(split_name) for split_name in ('train', 'val', 'test')}
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
def image_files(directory):
    return sorted(path for path in Path(directory).iterdir() if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES)

IMAGE_COUNTS = {name: len(image_files(path)) for name, path in SPLIT_DIRS.items()}
print('Dataset root:', DATASET_ROOT)
print('data.yaml:', DATA_YAML_PATH)
print('Raw split values:', {key: DATA_CONFIG.get(key) for key in ('train', 'val', 'test')})
print('Resolved split directories:', SPLIT_DIRS)
print('Image counts:', IMAGE_COUNTS)

# Strict paper-protocol guard: sai là dừng trước training.
assert IMAGE_COUNTS['train'] == EXPECTED_TRAIN_IMAGES, IMAGE_COUNTS
assert IMAGE_COUNTS['val'] == EXPECTED_VALID_IMAGES, IMAGE_COUNTS
assert IMAGE_COUNTS['test'] == EXPECTED_TEST_IMAGES, IMAGE_COUNTS

OUTPUT_DIR = Path('/content/drive/MyDrive/DetectedHelmets/paper2025_benchmark')
EXPERIMENT_NAME = 'paper2025_yolo26s_640_200ep'
EXPERIMENT_DIR = OUTPUT_DIR / EXPERIMENT_NAME
BEST_MODEL_PATH = EXPERIMENT_DIR / 'weights' / 'best.pt'
BASELINE_SUMMARY_PATH = OUTPUT_DIR / 'paper2025_yolo26s_summary.csv'
TEST_EVALUATED = False
TRAIN_CONFIG = {
    'epochs': 200, 'imgsz': 640, 'batch': 16, 'device': 0, 'workers': 4,
    'optimizer': 'auto', 'seed': 0, 'deterministic': True, 'pretrained': True,
    'amp': True, 'patience': 50, 'close_mosaic': 10,
    'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4,
    'translate': 0.1, 'scale': 0.5, 'fliplr': 0.5, 'mosaic': 1.0,
    'mixup': 0.0, 'cutmix': 0.0, 'copy_paste': 0.0,
}
RUNTIME_DATA_YAML_PATH = OUTPUT_DIR / 'resolved_data.yaml'
RUNTIME_DATA_CONFIG = dict(DATA_CONFIG)
RUNTIME_DATA_CONFIG.update({'path': str(DATASET_ROOT), 'train': str(SPLIT_DIRS['train']), 'val': str(SPLIT_DIRS['val']), 'test': str(SPLIT_DIRS['test']), 'nc': 3, 'names': CLASS_NAMES})
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with RUNTIME_DATA_YAML_PATH.open('w', encoding='utf-8') as handle: yaml.safe_dump(RUNTIME_DATA_CONFIG, handle, sort_keys=False, allow_unicode=True)
print('Ultralytics:', ultralytics.__version__)
print('Classes:', CLASS_NAMES)
print('Protocol:', TRAIN_CONFIG)

## Audit label và dry-run detection loader

Label được quét tối đa 1000 file. Nếu có row dài hơn 5 giá trị, notebook ghi nhận polygon source. Không convert file label. Ultralytics detection loader phải đọc được polygon và derive bounding boxes trong bộ nhớ; nếu không, notebook dừng.

In [ ]:
label_files = sorted((SPLIT_DIRS['train'].parent / 'labels').glob('*.txt'))
sample_files = random.Random(RANDOM_SEED).sample(label_files, min(1000, len(label_files)))
row_length_counts = Counter()
for label_path in sample_files:
    for raw_line in label_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if line:
            row_length_counts[len(line.split())] += 1

if not row_length_counts:
    raise RuntimeError('Không có label row để audit.')
HAS_POLYGON_LABELS = any(length > 5 for length in row_length_counts)
HAS_DETECTION_LABELS = any(length == 5 for length in row_length_counts)
if any(length not in (5,) and length <= 5 for length in row_length_counts):
    raise RuntimeError(f'Row label không hợp lệ: {row_length_counts}')
LABEL_MODE = 'polygon source, detection boxes derived by Ultralytics' if HAS_POLYGON_LABELS else 'YOLO detection rows'
print('Scanned label files:', len(sample_files))
print('Detection label rows:', row_length_counts.get(5, 0))
print('Polygon label rows:', sum(count for length, count in row_length_counts.items() if length > 5))
print('LABEL_MODE =', LABEL_MODE)

from ultralytics.data.dataset import YOLODataset

def detection_loader_dry_run(image_dir):
    # Constructor arguments are filtered for minor API differences, but no training is invoked.
    signature = inspect.signature(YOLODataset)
    optional = {
        'imgsz': IMG_SIZE, 'cache': False, 'augment': False, 'rect': False,
        'stride': 32, 'pad': 0.0, 'single_cls': False, 'classes': None,
        'fraction': 1.0, 'prefix': 'audit: ',
    }
    accepted = {key: value for key, value in optional.items() if key in signature.parameters}
    dataset = YOLODataset(str(image_dir), data={'nc': 3, 'names': CLASS_NAMES}, task='detect', **accepted)
    if len(dataset) == 0:
        raise RuntimeError(f'Loader không thấy ảnh tại {image_dir}')
    item = dataset[0]
    if 'bboxes' in item:
        bbox_count = len(item['bboxes'])
    elif 'instances' in item:
        bbox_count = len(item['instances'].bboxes)
    else:
        raise RuntimeError(f'Detection loader không trả bboxes/instances: {item.keys()}')
    if bbox_count == 0:
        raise RuntimeError('Dry-run loader trả 0 bounding boxes cho sample đầu tiên.')
    return len(dataset), item, bbox_count

train_loader_len, train_sample, train_bbox_count = detection_loader_dry_run(SPLIT_DIRS['train'])
valid_loader_len, valid_sample, valid_bbox_count = detection_loader_dry_run(SPLIT_DIRS['val'])
print('Detection loader dry-run:', {'train': train_loader_len, 'valid': valid_loader_len, 'train_sample_boxes': train_bbox_count, 'valid_sample_boxes': valid_bbox_count})
if HAS_POLYGON_LABELS:
    LABEL_MODE = 'polygon source, detection boxes derived by Ultralytics'
print('No polygon-to-bbox files were written; dataset remains unchanged.')

## Baseline training — YOLO26s, không custom module

Cell này chỉ chạy khi đổi `RUN_TRAINING = True` sau khi audit đã PASS. Output dùng `exist_ok=False` để không âm thầm tạo `_2`/`_3`.

In [ ]:
if RUN_TRAINING:
    from ultralytics import YOLO
    model = YOLO(MODEL_NAME)
    train_results = model.train(
        data=str(RUNTIME_DATA_YAML_PATH),
        project=str(OUTPUT_DIR), name=EXPERIMENT_NAME, exist_ok=False,
        val=True, **TRAIN_CONFIG
    )
    if not BEST_MODEL_PATH.exists():
        raise FileNotFoundError(f'best.pt ontbreekt: {BEST_MODEL_PATH}')
    print('Training klaar:', BEST_MODEL_PATH)
else:
    print('TRAINING DISABLED — audit-only notebook. Zet RUN_TRAINING=True om later bewust te trainen.')

## Validation chỉ trên `valid`

Không gọi `split='test'`. Metrics gồm tổng thể, per-class và confusion matrix do Ultralytics tạo trong thư mục experiment.

In [ ]:
validation_metrics = None
per_class_metrics = None
best_model = None
if RUN_VALIDATION:
    from ultralytics import YOLO
    if not BEST_MODEL_PATH.exists():
        raise FileNotFoundError(f'Chưa có best.pt: {BEST_MODEL_PATH}')
    best_model = YOLO(str(BEST_MODEL_PATH))
    metrics = best_model.val(
        data=str(RUNTIME_DATA_YAML_PATH), split='val', imgsz=IMG_SIZE, device=0,
        workers=4, plots=True, verbose=False
    )
    precision = float(metrics.box.mp)
    recall = float(metrics.box.mr)
    map50 = float(metrics.box.map50)
    map50_95 = float(metrics.box.map)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    rows = []
    for class_id, class_name in enumerate(CLASS_NAMES):
        p, r, ap50, ap50_95_class = metrics.box.class_result(class_id)
        p, r, ap50, ap50_95_class = map(float, (p, r, ap50, ap50_95_class))
        class_f1 = 2 * p * r / (p + r) if p + r else 0.0
        rows.append({'Class': class_name, 'Precision': p, 'Recall': r, 'F1': class_f1, 'AP50': ap50, 'AP50-95': ap50_95_class})
    per_class_metrics = pd.DataFrame(rows)
    validation_metrics = {'Precision': precision, 'Recall': recall, 'F1': f1, 'mAP50': map50, 'mAP50_95': map50_95}
    print('Overall validation:', validation_metrics)
    display(per_class_metrics)
    print('Confusion matrix:', EXPERIMENT_DIR / 'confusion_matrix.png')
else:
    print('VALIDATION DISABLED — chạy sau khi đã có best.pt và đặt RUN_VALIDATION=True.')

## Complexity và inference speed

Đo tối đa 50 ảnh validation sau warm-up. Không dùng folder test.

In [ ]:
complexity_metrics = {'Parameters': np.nan, 'GFLOPs': np.nan, 'Inference_ms': np.nan, 'FPS': np.nan, 'GPU_peak_VRAM_MB': np.nan}
if RUN_COMPLEXITY:
    if best_model is None:
        raise RuntimeError('Cần chạy validation trước complexity để load best.pt.')
    parameter_count = sum(parameter.numel() for parameter in best_model.model.parameters())
    info = best_model.info(verbose=False, imgsz=IMG_SIZE)
    gflops = np.nan
    if isinstance(info, dict):
        for key in ('GFLOPs', 'gflops', 'flops'):
            if key in info:
                gflops = float(info[key])
                break
    elif isinstance(info, (tuple, list)):
        for value in reversed(info):
            if isinstance(value, (int, float)) and float(value) > 0:
                gflops = float(value)
                break
    speed_images = image_files(SPLIT_DIRS['val'])[:50]
    if not speed_images:
        raise RuntimeError('Không có ảnh validation để đo tốc độ.')
    for image_path in speed_images[:min(2, len(speed_images))]:
        best_model.predict(source=str(image_path), imgsz=IMG_SIZE, device=0, save=False, verbose=False)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    elapsed = []
    for image_path in speed_images:
        if torch.cuda.is_available(): torch.cuda.synchronize()
        started = time.perf_counter()
        best_model.predict(source=str(image_path), imgsz=IMG_SIZE, device=0, save=False, verbose=False)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        elapsed.append(time.perf_counter() - started)
    inference_ms = float(np.mean(elapsed) * 1000)
    peak_vram = float(torch.cuda.max_memory_allocated() / (1024 ** 2)) if torch.cuda.is_available() else np.nan
    complexity_metrics = {'Parameters': parameter_count, 'GFLOPs': gflops, 'Inference_ms': inference_ms, 'FPS': 1000.0 / inference_ms, 'GPU_peak_VRAM_MB': peak_vram}
    print('Complexity:', complexity_metrics)
else:
    print('COMPLEXITY DISABLED — đặt RUN_COMPLEXITY=True sau validation nếu cần đo.')

## Save baseline summary và final audit

In [ ]:
if RUN_VALIDATION:
    per_class_lookup = per_class_metrics.set_index('Class')
    row = {
        'Model': 'YOLO26s', 'Dataset': EXPECTED_DATASET_NAME,
        'TrainImages': IMAGE_COUNTS['train'], 'ValidImages': IMAGE_COUNTS['val'],
        'imgsz': IMG_SIZE, 'epochs': TRAIN_CONFIG['epochs'], **validation_metrics,
        'Helmet_AP50': per_class_lookup.loc['helmet', 'AP50'],
        'NoHelmet_AP50': per_class_lookup.loc['nohelmet', 'AP50'],
        'Riders_AP50': per_class_lookup.loc['riders', 'AP50'],
        **complexity_metrics,
    }
    baseline_summary = pd.DataFrame([row])
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    baseline_summary.to_csv(BASELINE_SUMMARY_PATH, index=False)
    display(baseline_summary)
    print('Saved:', BASELINE_SUMMARY_PATH)
else:
    print('Summary chưa ghi vì RUN_VALIDATION=False.')

assert ultralytics.__version__ == ULTRALYTICS_VERSION
assert IMAGE_COUNTS['train'] == EXPECTED_TRAIN_IMAGES and IMAGE_COUNTS['val'] == EXPECTED_VALID_IMAGES
assert IMAGE_COUNTS['test'] == EXPECTED_TEST_IMAGES
assert CLASS_NAMES == EXPECTED_CLASS_NAMES
assert TRAIN_CONFIG['epochs'] == 200 and TRAIN_CONFIG['imgsz'] == 640 and TRAIN_CONFIG['batch'] == 16
assert TRAIN_CONFIG['seed'] == 0 and TRAIN_CONFIG['deterministic'] is True
assert not TEST_EVALUATED
print('Final audit: PASS')
print('Dataset audit included: YES | 3912/355 strict guard: YES | Test locked: YES')
print('Training executed in this creation session: NO')